<a href="https://colab.research.google.com/github/jeremiasiglesias-lgtm/green-hydrogen-nl/blob/main/notebooks/01_core_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Green hydrogen cost model
Levelised cost of hydrogen (LCOH) in €/kg for a grid-connected electrolyser in the Netherlands.

## Changelog
- 5 Nov 2026: core LCOH model (capex via CRF, O&M, electricity). Base case €5.73/kg.
- 6 Nov 2026: assumptions moved into one dictionary; added lcoh_breakdown() and checks.
- 6 Nov 2026: added economies of scale (scale_exponent, placeholder 1.0).
- 6 Nov 2026: sourced base values (capex 1950 €/kW, efficiency 54.6 kWh/kg, electricity 87 €/MWh, scale exponent 0.87).
- 6 Nov 2026: added stack replacement (48% of capex per replacement, 90,000 h stack life). No replacement at CF 0.5, so base LCOH unchanged at €7.96/kg (€8.37/kg with a 60,000 h stack).
- 7 Nov 2026: added water cost (10 L/kg demineralised water at €0.85/m³). Adds ~€0.01/kg; base LCOH €7.96 → €7.97/kg.
- 7 Nov 2026: added grid transport tariff (€150k/MW/yr, TenneT HV). Adds ~€1.87/kg; base LCOH €7.97 → €9.83/kg.
- 7 Nov 2026: added compression into the Dutch H2 network (~30 → 50 bar). Adds ~€0.23/kg; base LCOH €9.83 → €10.07/kg.

# 1. Assumptions

### Assumptions and sources (technology: alkaline, Netherlands, 2025 data)

| Assumption | Base | Low | High | Unit | Notes | Source | Year | Link |
|---|---|---|---|---|---|---|---|---|
| Capex | 1,950 | 1,680 | 2,210 | €/kW | Installed system, non-Chinese manufacturer; converted from USD 1,900–2,500/kW at 1.13 USD/€ | IEA Global Hydrogen Review 2026 (via Argus Media) | 2025 | https://www.argusmedia.com/en/news-and-insights/latest-market-news/2841836-green-h2-electrolyser-sector-enters-consolidation-iea |
| EUR/USD rate | 1.13 | – | – | USD/€ | 2025 annual average, ECB reference rate | Bank of Finland (ECB data) | 2025 | https://www.suomenpankki.fi/en/statistics/interest-rates-and-exchange-rates/exchange-rates/average-annual-exchange-rates-for-the-year-2025/ |
| Efficiency | 54.6 | 50.5 | 57.6 | kWh/kg | Full system. Base: BNEF 2026 benchmark. Low: IEA 66% LHV for 2023 (33.3 / 0.66 = 50.5). High: average of commercial alkaline models (mostly smaller units). Cross-check: US DOE system status 55 kWh/kg | BNEF via John Cockerill; IEA GHR 2024 Assumptions annex; Pietra et al. (Energies, 2021); US DOE | 2021–2025 | https://iea.blob.core.windows.net/assets/36017d2f-747b-4993-b06d-33209fe143fa/GHR24AssumptionsAnnex.pdf ; https://arts.units.it/retrieve/e2913fde-cecd-f688-e053-3705fe0a67e0/energies-14-05347-v2_compressed2.pdf ; https://www.energy.gov/node/4827230 |
| O&M | 3 | 2 | 5 | % of capex/yr | 2–3% in industry overview, 5% in a 2025 study | Decarbonisation Technology (PTQ); arXiv 2509.00136 | 2025 | https://ptqmagazines.digitalrefining.com/view/419891467/74/ ; https://arxiv.org/pdf/2509.00136 |
| Discount rate | 8 | 4 | 10 | % | 7% low-risk, 10% high-risk project; 4% with strong policy support | Andritz white paper; Agora Energiewende | 2025 | https://www.andritz.com/power-to-x-en/resources/wacc-influence-on-levelized-cost-of-green-hydrogen ; https://agora-energiewende.org/fileadmin/AutomaticFiles/1492/abb-06.pdf |
| Plant lifetime | 20 | 20 | 25 | years | Recent LCOH studies use 20 years (TU Delft, 200 MW NL) or 25 years (NESO; TU Delft, 10 MW alkaline) | TU Delft; NESO | 2023–2025 | https://repository.tudelft.nl/record/uuid:3642ff69-9719-4400-8246-5e3e4dc7f6a5 ; https://www.neso.energy/data-portal/levelised-cost-green-hydrogen/levelised_cost_of_green_hydrogen_modelling_2025_to_2050 |
| Electricity price | 87 | 77 | 96 | €/MWh | NL day-ahead annual averages after the crisis: 2024 €77, 2025 €87, 2023 €96. 2022 (€241) used as a stress test, not a range. Flat average; Milestone 3 replaces this | Electricity Maps; euenergy.live (ENTSO-E data) | 2023–2025 | https://www.electricitymaps.com/grid-in-review-2025/the-netherlands ; https://euenergy.live/netherlands/2023 ; https://euenergy.live/netherlands/2022 |
| Capacity factor | 0.5 | – | – | fraction | NOT SOURCED: depends on operating strategy, decided in Milestone 3 | – | – | – |
| Scale exponent | 0.87 | 0.76 | 1.0 | – | Alkaline >10 MW (0.76 = <10 MW); no extra scale benefit >100 MW | Energinet / Zauner et al., in arXiv 2104.04151 | 2021 | https://arxiv.org/pdf/2104.04151 |
| Stack lifetime | 90,000 | 50,000 | 90,000 | hours | High: manufacturer claims. Low: IEA's economic optimum considering degradation (stacks can reach 95,000 h) | Decarbonisation Technology (PTQ); IEA GHR 2024 Assumptions annex | 2024–2025 | https://iea.blob.core.windows.net/assets/36017d2f-747b-4993-b06d-33209fe143fa/GHR24AssumptionsAnnex.pdf |
| Stack replacement cost | 48 | – | – | % of capex | Single study (PEM); needs a second source | arXiv 2509.00136 | 2025 | https://arxiv.org/pdf/2509.00136 |
| Water use | 10 | 9 | 15 | L/kg H2 | Demineralised water bought in; 9 = stoichiometric minimum (2H2O → 2H2 + O2); 15 if purified on site | Chemistry; Gasworld | 2024 | https://www.gasworld.com/story/class-of-h2-module-7-ultra-pure-water-for-electrolysis/2141856.article.article/ |
| Water price | 0.85 | 0.85 | 2.19 | €/m³ | Base: NL demineralised water contract price (USD 0.905/m³, May 2023). High: average Dutch drinking water incl. tax (2023) | Intratec; Rabobank | 2023 | https://www.intratec.us/solutions/industry-economics-competitiveness/demineralized-water-cost/netherlands ; https://rabobank.com/knowledge/d011409185-the-price-of-water |
| Grid tariff | 150,000 | 116,000 | 150,000 | €/MW/yr | TenneT HV (base) vs EHV (low) transport tariff. The non-firm contract (57k, ~15% curtailment) is a commercial choice, so it is now a separate scenario, not part of the range | Modo Energy (TenneT tariffs); TNO 2024 | 2026 | https://modoenergy.com/research/en/nl-netherlands-bess-buildout-spreads-grid-connection-charges-battery-energy-storage-2026 |
| Compressor capex | 167 | 108 | 167 | €/(kg/day) | Pipeline compressor, sized on peak output. EU programme target, not market price | Clean Hydrogen JU call (2025) | 2025 | https://errin.eu/calls/demonstration-innovative-solutions-high-capacity-reliable-flexible-and-sustainable-hydrogen |
| Compressor O&M | 6 | – | – | % of compressor capex/yr | Single study | arXiv 2509.00136 | 2025 | https://arxiv.org/pdf/2509.00136 |
| Compression energy | 0.6 | 0.3 | 2.5 | kWh/kg | Design choice: ~30 → 50 bar into Dutch H2 network. Base: 25 → 85 bar uses ~1.7% of H2 energy (LHV). High: 30 → 200 bar | Neuman & Esser; Clean Hydrogen JU; Industry & Energy (Gasunie, 50 bar) | 2021–2025 | https://www.neuman-esser.com/en/news-media/magazine/hydrogen-transport-with-pipelines/ |
| Compressor lifetime | 14 | 14 | 20 | years | EU programme KPI: 14 years now, potential 20 by 2030. Replaced once within plant life | Clean Hydrogen JU call (2025) | 2025 | https://errin.eu/calls/demonstration-innovative-solutions-high-capacity-reliable-flexible-and-sustainable-hydrogen |

Still to source: grey H2 benchmark (Milestone 4).

In [4]:
base = {
    "capacity_kw": 10000,          # electrolyser size, kW
    "lifetime": 20,             # years
    "hours_per_year": 8760,       # 365 × 24
    "capex_per_kw": 1950,        # €/kW
    "discount_rate": 0.08,                # discount rate
    "om_fraction": 0.03,         # share of capex per year
    "efficiency": 54.6,            # kWh per kg H2
    "capacity_factor": 0.5,      # share of the year running
    "electricity_price": 87,     # €/MWh
    "reference_capacity_kw": 10_000,   # plant size the capex_per_kw figure applies to
    "scale_exponent": 0.87,             # 1.0 = no economies of scale (placeholder, to source)
    "stack_lifetime_h": 90_000,          # operating hours before the stack must be replaced (alkaline, manufacturer claim)
    "stack_replacement_fraction": 0.48,  # replacement cost as share of capex (arXiv 2509.00136)
    "water_l_per_kg": 10,          # litres demineralised water per kg H2 (9 L stoichiometric + margin)
    "water_price_per_m3": 0.85,    # €/m³, NL demineralised water contract price (Intratec, May 2023)
    "grid_tariff_per_mw_year": 150_000,   # €/MW/yr, TenneT HV transport tariff; 57k with non-firm contract
    "compressor_capex_per_kg_day": 167,   # €/(kg/day) peak capacity, pipeline compressor (Clean Hydrogen JU)
    "compressor_om_fraction": 0.06,       # share of compressor capex per year (arXiv 2509.00136)
    "compression_kwh_per_kg": 0.6,        # kWh/kg, ~30 -> 50 bar into Dutch H2 network (Neuman & Esser: 25->85 bar)
    "compressor_lifetime_years": 14,      # technical lifetime (Clean Hydrogen JU)
}

In [5]:
# --- Sourced low/high ranges for each uncertain input (from the assumptions table) ---
# A dictionary whose values are TUPLES: (low, high).
# A tuple is like a list in round brackets, but it can't be changed afterwards.
# That's good here: a range is a fixed pair of numbers.
# --- Sourced low/high ranges for each uncertain input (see assumptions table) ---
ranges = {
    "capex_per_kw":                (1680, 2210),      # €/kW
    "efficiency":                  (50.5, 57.6),      # kWh/kg   (was 47-73)
    "om_fraction":                 (0.02, 0.05),      # share of capex per year
    "discount_rate":               (0.04, 0.10),      # fraction
    "lifetime":                    (20, 25),          # years    (was 10-30)
    "electricity_price":           (77, 96),          # €/MWh    (was 77-87)
    "scale_exponent":              (0.76, 1.0),       # -  kept for the dashboard, skipped in the tornado
    "stack_lifetime_h":            (50_000, 90_000),  # hours    (was 60,000-90,000)
    "water_l_per_kg":              (9, 15),           # L/kg H2
    "water_price_per_m3":          (0.85, 2.19),      # €/m³
    "grid_tariff_per_mw_year":     (116_000, 150_000),# €/MW/yr  (was 57,000-150,000)
    "compressor_capex_per_kg_day": (108, 167),        # €/(kg/day)
    "compression_kwh_per_kg":      (0.3, 2.5),        # kWh/kg
    "compressor_lifetime_years":   (14, 20),          # years
}

# capacity_factor is left out on purpose: it has no sourced range yet (Milestone 3).

# 2. Functions

In [6]:
def crf(rate, lifetime):
    growth = (1 + rate) ** lifetime
    return rate * growth / (growth - 1)

def lcoh_breakdown(a):
    lifetime = a["lifetime"]
    capacity_kw = a["capacity_kw"]
    hours_per_year = a["hours_per_year"]
    energy_kwh = capacity_kw * hours_per_year * a["capacity_factor"]
    h2_kg = energy_kwh / a["efficiency"]
    size_ratio = capacity_kw / a["reference_capacity_kw"]
    capex_total = a["reference_capacity_kw"]*a["capex_per_kw"] * size_ratio ** a["scale_exponent"]


    stack_life_years = a["stack_lifetime_h"] / (hours_per_year * a["capacity_factor"])


    replacement_pv = 0
    year = stack_life_years
    while year < lifetime:
        cost = capex_total * a["stack_replacement_fraction"]
        replacement_pv += cost / (1 + a["discount_rate"]) ** year
        year +=  stack_life_years

    stack_per_year = replacement_pv * crf(a["discount_rate"], lifetime)

    capex_per_year = capex_total * crf(a["discount_rate"], lifetime)
    om_per_year = capex_total * a["om_fraction"]
    electricity_per_year = (energy_kwh / 1000) * a["electricity_price"]
    water_per_year = h2_kg * (a["water_l_per_kg"] / 1000) * a["water_price_per_m3"]   # L -> m³
    grid_per_year = (capacity_kw/1000) * a["grid_tariff_per_mw_year"]        # kW -> MW

    peak_kg_per_day = capacity_kw / a["efficiency"] * 24                    # kg/day at full power
    compressor_capex = peak_kg_per_day * a["compressor_capex_per_kg_day"]
    compressor_replacement_pv = 0
    year = a["compressor_lifetime_years"]
    while year < lifetime:
        compressor_replacement_pv += compressor_capex / (1 + a["discount_rate"]) ** year
        year += a["compressor_lifetime_years"]

    compression_energy_cost = h2_kg * a["compression_kwh_per_kg"] / 1000 * a["electricity_price"]   # kWh -> MWh
    compression_per_year = ((compressor_capex + compressor_replacement_pv) * crf(a["discount_rate"], lifetime)
                            + compressor_capex * a["compressor_om_fraction"]
                            + compression_energy_cost)

    return {
        "capex": capex_per_year/h2_kg,
        "om": om_per_year/h2_kg,
        "electricity": electricity_per_year/h2_kg,
        "stack": stack_per_year / h2_kg,
        "water": water_per_year / h2_kg,
        "grid": grid_per_year / h2_kg,
        "compression": compression_per_year / h2_kg,
        "total": (capex_per_year + om_per_year + electricity_per_year+ stack_per_year + water_per_year + grid_per_year + compression_per_year)/h2_kg,
    }

def lcoh_from(a):
    """LCOH in €/kg from the dictionary of assumptions."""
    return lcoh_breakdown(a)["total"]

# 3. Base case result

In [7]:
print(f"LCOH: €{lcoh_from(base):.2f} per kg")

result = lcoh_breakdown(base)
for part, value in result.items():
    print(f"{part:12} €{value:.2f}/kg")



LCOH: €10.07 per kg
capex        €2.48/kg
om           €0.73/kg
electricity  €4.75/kg
stack        €0.00/kg
water        €0.01/kg
grid         €1.87/kg
compression  €0.23/kg
total        €10.07/kg


### Test: 60,000-hour stack

In [8]:
test = base.copy()
test["stack_lifetime_h"] = 60_000
for part, value in lcoh_breakdown(test).items():
    print(f"{part:12} €{value:.2f}/kg")

capex        €2.48/kg
om           €0.73/kg
electricity  €4.75/kg
stack        €0.41/kg
water        €0.01/kg
grid         €1.87/kg
compression  €0.23/kg
total        €10.48/kg


Running more hours wears out the stack faster and adds replacement costs, but spreading the fixed costs over more hydrogen saves far more, so with a flat electricity price, running more always wins.

# 4. Checks

In [9]:
r = lcoh_breakdown(base)


assert abs(r["capex"] + r["om"] + r["electricity"] + r["stack"] + r["water"] + r["grid"] + r["compression"]- r["total"]) < 0.001


assert abs(lcoh_from(base) - r["total"]) < 0.001

big = base.copy()
big["capacity_kw"] = 100_000
big["scale_exponent"] = 1.0
assert abs(lcoh_from(big) - lcoh_from(base)) < 0.001

print("All checks passed")



All checks passed
